# Сравнение стратегий JOIN

In [2]:
import time
import statistics

from pathlib import Path
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("SCD2_Homework")
         .master("spark://spark-master:7077")
         .getOrCreate())

# Для корректного сравнения временных меток используем UTC
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Сохраняем исходные настройки Spark для автоматического JOIN
CONF_KEYS = [
    "spark.sql.adaptive.enabled",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.shuffle.partitions"
]

ORIGINAL_CONF = {
    key: spark.conf.get(key)
    for key in CONF_KEYS
}

# Читаем уже построенную и сохранённую SCD2-витрину
df_scd2 = spark.read.parquet("s3a://dwh/scd2/user_events/")

# Оставляем только необходимые для задания столбцы
df_scd2 = df_scd2.select(
    "user_id",
    "user_fio",
    "user_email",
    "user_login",
    "valid_from",
    "valid_to"
)

df_scd2.printSchema()

root
 |-- user_id: long (nullable = true)
 |-- user_fio: string (nullable = true)
 |-- user_email: string (nullable = true)
 |-- user_login: string (nullable = true)
 |-- valid_from: timestamp (nullable = true)
 |-- valid_to: timestamp (nullable = true)



### Загрузка справочника компаний

In [3]:
campaigns_path = "/home/jovyan/work/data/task1/campaigns.csv"

df_campaigns_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "FAILFAST")
    .csv(campaigns_path)
)

# Явно задаём типы столбцов и преобразуем send_at в timestamp
df_campaigns = df_campaigns_raw.select(
    F.col("campaign_id").cast("int").alias("campaign_id"),
    F.lower(F.trim(F.col("email_domain"))).alias("email_domain"),
    F.to_timestamp(
        F.trim(F.col("send_at")),
        "yyyy-MM-dd'T'HH:mm:ss'Z'"
    ).alias("send_at"),
    F.col("campaign_name").cast("string").alias("campaign_name")
)

df_campaigns.show(30, truncate=False)
df_campaigns.printSchema()

campaign_count = df_campaigns.count()

print("Количество кампаний:", campaign_count)
print("Количество доменов:", df_campaigns.select("email_domain").distinct().count())

invalid_campaigns = df_campaigns.filter(
    F.col("campaign_id").isNull() |
    F.col("email_domain").isNull() |
    F.col("send_at").isNull() |
    F.col("campaign_name").isNull()
).count()
assert invalid_campaigns == 0, "обнаружены NULL или ошибки преобразования"

duplicate_campaigns = (
    df_campaigns
    .groupBy("campaign_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
assert duplicate_campaigns == 0, "обнаружены повторяющиеся campaign_id"

+-----------+---------------+-------------------+-------------+
|campaign_id|email_domain   |send_at            |campaign_name|
+-----------+---------------+-------------------+-------------+
|1          |mail.example   |2026-02-04 09:25:59|Campaign_0001|
|2          |inbox.example  |2026-01-20 07:00:16|Campaign_0002|
|3          |webmail.example|2026-02-15 01:56:37|Campaign_0003|
|4          |webmail.example|2026-02-17 23:41:49|Campaign_0004|
|5          |post.example   |2026-01-07 02:27:37|Campaign_0005|
|6          |mail.example   |2026-02-12 14:57:48|Campaign_0006|
|7          |example.com    |2026-02-14 21:44:10|Campaign_0007|
|8          |example.com    |2026-01-14 12:54:19|Campaign_0008|
|9          |example.com    |2026-02-09 16:16:48|Campaign_0009|
|10         |mail.example   |2026-01-27 07:16:13|Campaign_0010|
|11         |mail.example   |2026-01-23 07:18:34|Campaign_0011|
|12         |inbox.example  |2026-01-20 06:37:56|Campaign_0012|
|13         |mail.example   |2026-01-14 

### Реализация четырёх вариантов JOIN

In [4]:
STRATEGY_ORDER = [
    "auto",
    "sort_merge",
    "broadcast_hash",
    "shuffled_hash"
]


# Настраивает Spark в соответствии с выбранной стратегией
def configure_strategy(strategy):
    if strategy == "auto":
        # Возвращаем исходные настройки Spark
        for key, value in ORIGINAL_CONF.items():
            spark.conf.set(key, value)

    else:
        # Для трёх явно заданных стратегий используем одинаковые условия
        spark.conf.set("spark.sql.adaptive.enabled", "false")
        spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
        spark.conf.set("spark.sql.shuffle.partitions", "16")


# Строит аудиторию кампаний с использованием выбранной стратегии JOIN
def build_campaign_audience(strategy):
    campaigns = df_campaigns

    if strategy == "sort_merge":
        campaigns = campaigns.hint("MERGE")
    elif strategy == "broadcast_hash":
        campaigns = F.broadcast(campaigns)
    elif strategy == "shuffled_hash":
        campaigns = campaigns.hint("SHUFFLE_HASH")
    elif strategy != "auto":
        raise ValueError(f"Неизвестная стратегия: {strategy}")

    users = df_scd2.alias("u")
    campaigns = campaigns.alias("c")

    # Извлекаем домен электронной почты пользователя
    user_email_domain = F.lower(
        F.trim(
            F.split(F.col("u.user_email"), "@").getItem(1)
        )
    )

    join_condition = (
        # Историческая версия должна действовать на момент отправки
        (F.col("u.valid_from") <= F.col("c.send_at")) &
        (
            F.col("u.valid_to").isNull() |
            (F.col("c.send_at") < F.col("u.valid_to"))
        ) &
        # Домен электронной почты должен совпадать с доменом кампании
        (user_email_domain == F.col("c.email_domain"))
    )

    result = (
        users
        .join(
            campaigns,
            join_condition,
            "inner"
        )
        .select(
            F.col("c.campaign_id"),
            F.col("c.campaign_name"),
            F.col("c.send_at"),
            F.col("c.email_domain"),
            F.col("u.user_id"),
            F.col("u.user_fio"),
            F.col("u.user_email"),
            F.col("u.user_login")
        )
    )

    return result

### Проверка корректности результатов

In [5]:
results = {}
row_counts = {}

for strategy in STRATEGY_ORDER:
    configure_strategy(strategy)
    print(f"Вычисление результата: {strategy}")

    df_result = build_campaign_audience(strategy)

    results[strategy] = df_result
    row_counts[strategy] = df_result.count()
    print(f"{strategy}: {row_counts[strategy]:,} строк")


# Используем автоматический join как эталон
df_reference = results["auto"]
for strategy in STRATEGY_ORDER[1:]:
    df_candidate = results[strategy]

    assert (
        df_reference.schema == df_candidate.schema
    ), f"Различаются схемы: {strategy}"

    # Проверяем отсутствие строк, которые есть только в одном результате
    left_difference = (
        df_reference
        .exceptAll(df_candidate)
        .limit(1)
        .count()
    )

    right_difference = (
        df_candidate
        .exceptAll(df_reference)
        .limit(1)
        .count()
    )

    is_equal = (
        left_difference == 0 and
        right_difference == 0
    )

    print(
        f"{strategy}: {'Результаты совпадают' if is_equal else 'ОШИБКА'}"
    )

    assert is_equal, f"Результаты не совпадают: {strategy}"


# Дополнительно проверяем уникальность пары campaign_id + user_id
duplicate_pairs = (
    df_reference
    .groupBy("campaign_id", "user_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_pair_count = duplicate_pairs.count()

print("\nКоличество повторяющихся пар campaign_id + user_id:",
      duplicate_pair_count)

if duplicate_pair_count > 0:
    duplicate_pairs.show(20, truncate=False)

assert duplicate_pair_count == 0, "Обнаружены повторяющиеся пары кампании и пользователя"

Вычисление результата: auto
auto: 500,233 строк
Вычисление результата: sort_merge
sort_merge: 500,233 строк
Вычисление результата: broadcast_hash
broadcast_hash: 500,233 строк
Вычисление результата: shuffled_hash
shuffled_hash: 500,233 строк
sort_merge: Результаты совпадают
broadcast_hash: Результаты совпадают
shuffled_hash: Результаты совпадают

Количество повторяющихся пар campaign_id + user_id: 0


### Сохранение физических планов

In [6]:
OUTPUT_ROOT = "s3a://dwh/task1/join_benchmarks"
OUTPUT_PATHS = {
    strategy: f"{OUTPUT_ROOT}/{strategy}"
    for strategy in STRATEGY_ORDER
}
PLAN_DIR = Path("/home/jovyan/work/task1_plans")
PLAN_DIR.mkdir(parents=True, exist_ok=True)
PHYSICAL_PLANS = {}


for strategy in STRATEGY_ORDER:
    configure_strategy(strategy)
    df_result = build_campaign_audience(strategy)
    spark.sparkContext.setJobGroup(
        f"task1_warmup_{strategy}",
        f"Task 1 warmup: {strategy}"
    )

    try:
        df_result.write.mode("overwrite").parquet(OUTPUT_PATHS[strategy])
    finally:
        spark.sparkContext.setLocalProperty("spark.jobGroup.id", None)
        spark.sparkContext.setLocalProperty("spark.job.description", None)

    # После выполнения действия получаем фактически использованный план
    plan_text = df_result._jdf.queryExecution().executedPlan().toString()

    PHYSICAL_PLANS[strategy] = plan_text

    plan_file = PLAN_DIR / f"{strategy}_plan.txt"
    plan_file.write_text(plan_text, encoding="utf-8")

    print("Физический план:")
    print(plan_text)

    print("\nПлан сохранён:", plan_file)

    expected_operators = {
        "sort_merge": "SortMergeJoin",
        "broadcast_hash": "BroadcastHashJoin",
        "shuffled_hash": "ShuffledHashJoin"
    }

    if strategy in expected_operators:
        expected = expected_operators[strategy]

        if expected not in plan_text:
            print(
                f"ВНИМАНИЕ: ожидался {expected}, "
                "но он не найден в физическом плане."
            )

Физический план:
AdaptiveSparkPlan isFinalPlan=false
+- Project [campaign_id#67, campaign_name#62, send_at#69, email_domain#68, user_id#21L, user_fio#22, user_email#23, user_login#24]
   +- BroadcastHashJoin [lower(trim(split(user_email#23, @, -1)[1], None))], [email_domain#68], Inner, BuildRight, ((valid_from#25 <= send_at#69) AND (isnull(valid_to#26) OR (send_at#69 < valid_to#26))), false
      :- Filter isnotnull(valid_from#25)
      :  +- FileScan parquet [user_id#21L,user_fio#22,user_email#23,user_login#24,valid_from#25,valid_to#26] Batched: true, DataFilters: [isnotnull(valid_from#25)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[s3a://dwh/scd2/user_events], PartitionFilters: [], PushedFilters: [IsNotNull(valid_from)], ReadSchema: struct<user_id:bigint,user_fio:string,user_email:string,user_login:string,valid_from:timestamp,va...
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[1, string, true]),false), [plan_id=2846]
         +- Project [cast(campaign_

По сохранённым планам:
- sort_merge использует SortMergeJoin
- broadcast_hash использует BroadcastHashJoin
- shuffled_hash использует ShuffledHashJoin
- в плане auto виден BroadcastHashJoin

У плана auto в выводе стоит AdaptiveSparkPlan isFinalPlan=false. Это не подтверждает конечный план после AQE. Для выполнения требования о конечном плане нужно открыть Spark UI приложения, перейти в SQL / Queries, найти запрос автоматического варианта и сохранить финальный план после выполнения. Остальные три варианта выполняются с отключённым AQE, их показанные операторы соответствуют заданным hints.

### Измерение времени выполнения

In [7]:
import pandas as pd

RUNS = 5
benchmark_results = []
for run_number in range(1, RUNS + 1):

    # Меняем порядок стратегий в каждом раунде
    shift = (run_number - 1) % len(STRATEGY_ORDER)

    current_order = (
        STRATEGY_ORDER[shift:] +
        STRATEGY_ORDER[:shift]
    )

    print(f"РАУНД {run_number}")
    print("Порядок:", current_order)

    for strategy in current_order:
        configure_strategy(strategy)
        df_result = build_campaign_audience(strategy)
        job_group_id = f"task1_{strategy}_run_{run_number}"
        spark.sparkContext.setJobGroup(
            job_group_id,
            f"Task 1 benchmark: {strategy}, run {run_number}"
        )

        start_time = time.perf_counter()
        try:
            df_result.write \
                .mode("overwrite") \
                .parquet(OUTPUT_PATHS[strategy])
        finally:
            elapsed = time.perf_counter() - start_time
            spark.sparkContext.setLocalProperty("spark.jobGroup.id", None)
            spark.sparkContext.setLocalProperty("spark.job.description", None)
        benchmark_results.append({
            "strategy": strategy,
            "run": run_number,
            "time_seconds": elapsed,
            "row_count": row_counts[strategy],
            "output_path": OUTPUT_PATHS[strategy]
        })

        print(f"{strategy}: {elapsed:.2f} секунд")


# Возвращаем исходные настройки Spark
for key, value in ORIGINAL_CONF.items():
    spark.conf.set(key, value)

РАУНД 1
Порядок: ['auto', 'sort_merge', 'broadcast_hash', 'shuffled_hash']
auto: 6.87 секунд
sort_merge: 8.01 секунд
broadcast_hash: 5.82 секунд
shuffled_hash: 7.98 секунд
РАУНД 2
Порядок: ['sort_merge', 'broadcast_hash', 'shuffled_hash', 'auto']
sort_merge: 7.99 секунд
broadcast_hash: 6.83 секунд
shuffled_hash: 7.64 секунд
auto: 5.85 секунд
РАУНД 3
Порядок: ['broadcast_hash', 'shuffled_hash', 'auto', 'sort_merge']
broadcast_hash: 5.79 секунд
shuffled_hash: 7.70 секунд
auto: 5.59 секунд
sort_merge: 7.91 секунд
РАУНД 4
Порядок: ['shuffled_hash', 'auto', 'sort_merge', 'broadcast_hash']
shuffled_hash: 7.72 секунд
auto: 5.64 секунд
sort_merge: 7.86 секунд
broadcast_hash: 5.53 секунд
РАУНД 5
Порядок: ['auto', 'sort_merge', 'broadcast_hash', 'shuffled_hash']
auto: 5.58 секунд
sort_merge: 8.57 секунд
broadcast_hash: 5.71 секунд
shuffled_hash: 7.74 секунд


### Обработка результатов

In [8]:
df_benchmark = pd.DataFrame(benchmark_results)
display(df_benchmark.sort_values(["run", "strategy"]))


summary_rows = []
for strategy in STRATEGY_ORDER:
    strategy_times = (
        df_benchmark[
            df_benchmark["strategy"] == strategy
        ]["time_seconds"]
        .tolist()
    )
    summary_rows.append({
        "strategy": strategy,
        "min_seconds": round(min(strategy_times), 2),
        "median_seconds": round(statistics.median(strategy_times), 2),
        "max_seconds": round(max(strategy_times), 2),
        "row_count": row_counts[strategy],
        "output_path": OUTPUT_PATHS[strategy]
    })


df_summary = pd.DataFrame(summary_rows)
display(df_summary)

,strategy,run,time_seconds,row_count,output_path
0,auto,1,6.873509,500233,s3a://dwh/task1/join_benchmarks/auto
2,broadcast_hash,1,5.819697,500233,s3a://dwh/task1/join_benchmarks/broadcast_hash
3,shuffled_hash,1,7.982415,500233,s3a://dwh/task1/join_benchmarks/shuffled_hash
1,sort_merge,1,8.012262,500233,s3a://dwh/task1/join_benchmarks/sort_merge
7,auto,2,5.852054,500233,s3a://dwh/task1/join_benchmarks/auto
5,broadcast_hash,2,6.832691,500233,s3a://dwh/task1/join_benchmarks/broadcast_hash
6,shuffled_hash,2,7.644940,500233,s3a://dwh/task1/join_benchmarks/shuffled_hash
4,sort_merge,2,7.990246,500233,s3a://dwh/task1/join_benchmarks/sort_merge
10,auto,3,5.590233,500233,s3a://dwh/task1/join_benchmarks/auto
8,broadcast_hash,3,5.792182,500233,s3a://dwh/task1/join_benchmarks/broadcast_hash


,strategy,min_seconds,median_seconds,max_seconds,row_count,output_path
0,auto,5.58,5.64,6.87,500233,s3a://dwh/task1/join_benchmarks/auto
1,sort_merge,7.86,7.99,8.57,500233,s3a://dwh/task1/join_benchmarks/sort_merge
2,broadcast_hash,5.53,5.79,6.83,500233,s3a://dwh/task1/join_benchmarks/broadcast_hash
3,shuffled_hash,7.64,7.72,7.98,500233,s3a://dwh/task1/join_benchmarks/shuffled_hash


На этом стенде минимальная медиана у автоматического варианта - 5.64 с. Явный Broadcast Hash Join близок к нему - 5.79 с; диапазоны измерений сильно пересекаются, поэтому разницу в 0.15 с нельзя считать каким-то преимуществом. Sort Merge Join и Shuffled Hash Join оказались медленнее - 7.99 и 7.72 с соответственно.

Spark без hints выбрал BroadcastHashJoin в сохранённом плане. Явный broadcast не изменил тип JOIN, поскольку Spark и без него выбрал broadcast для небольшого справочника из 25 кампаний.